In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("Key loaded:", GROQ_API_KEY[:10], "...")

Key loaded: gsk_aAEj3D ...


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

os.chdir(r"C:\Users\varun\OneDrive\Desktop\ai_Engineer_journey")
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY").strip()  # ← strip removes invisible characters

print("Key length:", len(GROQ_API_KEY))

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=GROQ_API_KEY
)

response = llm.invoke("Say hello in one sentence")
print(response.content)

Key starts with: gsk_
Key length: 56
Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import os
import json

os.chdir(r"C:\Users\varun\OneDrive\Desktop\ai_Engineer_journey")
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY").strip()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=GROQ_API_KEY
)

print("Setup complete. LLM ready.")

ModuleNotFoundError: No module named 'langchain.text_splitters'

In [ ]:
%pip install langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [12]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables.history import RunnableWithMessageHistory
from dotenv import load_dotenv
import os
import json

os.chdir(r"C:\Users\varun\OneDrive\Desktop\ai_Engineer_journey")
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY").strip()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=GROQ_API_KEY
)

print("Setup complete. LLM ready.")

Setup complete. LLM ready.


In [13]:
template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Answer concisely in 3-4 sentences."),
    ("human", "{question}")
])

chain = template | llm | StrOutputParser()

# Test 1 — ML domain
response1 = chain.invoke({
    "domain": "machine learning",
    "question": "What is overfitting?"
})
print("=== ML Answer ===")
print(response1)
print()

# Test 2 — DSA domain
response2 = chain.invoke({
    "domain": "data structures",
    "question": "When should I use a hashmap?"
})
print("=== DSA Answer ===")
print(response2)

=== ML Answer ===
Overfitting occurs when a machine learning model is too complex and learns the noise in the training data, resulting in poor performance on new, unseen data. This happens when the model is overly specialized to the training dataset and fails to generalize well to other data. As a result, the model's accuracy on the training data is high, but its accuracy on test data is low. Regularization techniques can help prevent overfitting by simplifying the model.

=== DSA Answer ===
Use a hashmap when you need to store and retrieve data efficiently by a unique key, such as caching, configuration data, or indexing large datasets. Hashmaps provide fast lookups, insertions, and deletions, with an average time complexity of O(1). They are particularly useful when you need to quickly retrieve a value based on a specific key or when you need to keep track of a large number of unique items. This makes them ideal for applications like dictionaries, caches, and database indexing.


In [14]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful AI engineering tutor. 
    You explain concepts clearly and remember the full conversation."""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | llm | StrOutputParser()

session_store = {}

def get_session_history(session_id: str):
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

config = {"configurable": {"session_id": "student_001"}}

print("Turn 1:")
r1 = chain_with_history.invoke(
    {"input": "What is a transformer in AI?"},
    config=config
)
print(r1)
print()

print("Turn 2:")
r2 = chain_with_history.invoke(
    {"input": "How does it relate to attention mechanisms?"},
    config=config
)
print(r2)
print()

print("Turn 3:")
r3 = chain_with_history.invoke(
    {"input": "Can you summarize what we just discussed?"},
    config=config
)
print(r3)
print()

# Show history
print("=== Message History ===")
messages = session_store['student_001'].messages
print(f"Total messages in history: {len(messages)}")
for msg in messages:
    print(f"[{type(msg).__name__}]: {msg.content[:80]}...")

Turn 1:
The transformer is a fundamental concept in artificial intelligence (AI), specifically in the field of natural language processing (NLP) and deep learning.

A transformer is a type of neural network architecture introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017. It's primarily designed for sequence-to-sequence tasks, such as machine translation, text summarization, and chatbots.

The transformer model relies heavily on self-attention mechanisms, which allow it to weigh the importance of different input elements (like words or characters) relative to each other. This is different from traditional recurrent neural networks (RNNs) and convolutional neural networks (CNNs), which use recurrent connections or convolutional layers to process input sequences.

The key components of a transformer model are:

1. **Self-Attention Mechanism**: This allows the model to attend to different parts of the input sequence simultaneously and weigh their importance.
2. *

In [15]:
document_text = """
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed.
It focuses on developing computer programs that can access data and use it
to learn for themselves.

Deep learning is a subset of machine learning that uses neural networks 
with many layers. These deep neural networks can learn representations 
of data with multiple levels of abstraction.

Natural language processing is a branch of AI that deals with the interaction 
between computers and humans through natural language. The ultimate objective 
is to read, decipher, understand, and make sense of human language.

Reinforcement learning is an area of machine learning concerned with how 
software agents ought to take actions in an environment to maximize cumulative reward.
"""

doc = Document(page_content=document_text, metadata={"source": "ai_concepts"})

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    length_function=len
)

chunks = splitter.split_documents([doc])

print(f"Original document: {len(document_text)} characters")
print(f"Number of chunks: {len(chunks)}")
print()

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk.page_content)} chars):")
    print(chunk.page_content)
    print()

Original document: 833 characters
Number of chunks: 6

Chunk 1 (151 chars):
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed.

Chunk 2 (99 chars):
It focuses on developing computer programs that can access data and use it
to learn for themselves.

Chunk 3 (189 chars):
Deep learning is a subset of machine learning that uses neural networks 
with many layers. These deep neural networks can learn representations 
of data with multiple levels of abstraction.

Chunk 4 (156 chars):
Natural language processing is a branch of AI that deals with the interaction 
between computers and humans through natural language. The ultimate objective

Chunk 5 (67 chars):
is to read, decipher, understand, and make sense of human language.

Chunk 6 (160 chars):
Reinforcement learning is an area of machine learning concerned with how 
software agents ought to take actions in an environment to maximize cumulat

In [16]:
def simple_rag(query, chunks, llm):
    # Find relevant chunks using keyword matching
    relevant_chunks = []
    query_words = set(query.lower().split())

    for chunk in chunks:
        chunk_words = set(chunk.page_content.lower().split())
        overlap = len(query_words & chunk_words)
        if overlap > 0:
            relevant_chunks.append((overlap, chunk.page_content))

    relevant_chunks.sort(reverse=True)
    context = "\n\n".join([c[1] for c in relevant_chunks[:2]])

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a helpful assistant. Answer the question 
        using ONLY the provided context. If the answer is not in the 
        context, say 'I don't have information about that.'
        
        Context:
        {context}"""),
        ("human", "{question}")
    ])

    chain = prompt | llm | StrOutputParser()

    return chain.invoke({
        "context": context,
        "question": query
    })

queries = [
    "What is deep learning?",
    "How does reinforcement learning work?",
    "What is the relationship between ML and AI?",
    "What is quantum computing?"
]

for query in queries:
    print(f"Q: {query}")
    print(f"A: {simple_rag(query, chunks, llm)}")
    print()

Q: What is deep learning?
A: Deep learning is a subset of machine learning that uses neural networks with many layers. These deep neural networks can learn representations of data with multiple levels of abstraction.

Q: How does reinforcement learning work?
A: Reinforcement learning works by having software agents take actions in an environment to maximize cumulative reward.

Q: What is the relationship between ML and AI?
A: I don't have information about that.

Q: What is quantum computing?
A: I don't have information about that.



In [17]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("""
Analyze this ML concept and return a JSON object with these exact keys:
- concept: the name of the concept
- difficulty: easy/medium/hard  
- prerequisites: list of 2-3 prerequisite concepts
- use_cases: list of 2-3 real world use cases
- one_line_summary: single sentence explanation

Concept: {concept}

Return ONLY valid JSON, no other text, no markdown backticks.
""")

chain = template | llm | StrOutputParser()

concepts = ["gradient descent", "attention mechanism", "random forest"]

for concept in concepts:
    raw = chain.invoke({"concept": concept})
    try:
        parsed = json.loads(raw)
        print(f"Concept:    {parsed['concept']}")
        print(f"Difficulty: {parsed['difficulty']}")
        print(f"Summary:    {parsed['one_line_summary']}")
        print(f"Use cases:  {', '.join(parsed['use_cases'])}")
        print()
    except json.JSONDecodeError:
        print(f"Failed to parse JSON for {concept}")
        print(raw)
        print()

Concept:    gradient descent
Difficulty: medium
Summary:    Gradient descent is an optimization algorithm used to minimize the loss function in machine learning models by iteratively adjusting the model's parameters in the direction of the negative gradient.
Use cases:  image classification, natural language processing, recommender systems

Concept:    attention mechanism
Difficulty: hard
Summary:    The attention mechanism is a technique used in deep learning models to focus on specific parts of the input data when generating outputs.
Use cases:  machine translation, question answering, image captioning

Concept:    random forest
Difficulty: medium
Summary:    Random forest is an ensemble learning method that combines multiple decision trees to improve the accuracy and robustness of predictions.
Use cases:  predicting customer churn, classifying medical images, forecasting stock prices

